In [16]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Embedding

In [17]:
df = pd.read_csv("spam.csv", encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:\n", df['label'].value_counts())

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['label'])

Dataset Shape: (5572, 2)

Class Distribution:
 label
ham     4825
spam     747
Name: count, dtype: int64


In [18]:
x_train,x_test,y_train,y_test=train_test_split(
    df['text'],df['label'],test_size=0.2,random_state=42,stratify=df['label']
)

MAX_WORDS=5000
MAX_LEN=50
EMBEDDING_DIM=64

tokenizer=Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

x_train_seq=tokenizer.texts_to_sequences(x_train)
x_test_seq=tokenizer.texts_to_sequences(x_test)

In [19]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM
    ),
        GRU(
        units=64,
        dropout=0.2,
        recurrent_dropout=0.2,
        return_sequences=False
        ),

        Dense(units=32, activation='relu'),
    Dropout(0.3),
    Dense(units=1, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ (None, 50, 64)              │         320,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_1 (GRU)                          │ (None, 64)                  │          24,960 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 347,073 (1.32 MB)

 Trainable params: 347,073 (1.32 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
X_train_pad = pad_sequences(x_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 63s 291ms/step - accuracy: 0.8571 - loss: 0.4584 - val_accuracy: 0.8453 - val_loss: 0.4372
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 18s 290ms/step - accuracy: 0.8681 - loss: 0.4052 - val_accuracy: 0.8453 - val_loss: 0.4341
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 211ms/step - accuracy: 0.8681 - loss: 0.4088 - val_accuracy: 0.8453 - val_loss: 0.4362
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 21s 222ms/step - accuracy: 0.8681 - loss: 0.3992 - val_accuracy: 0.8453 - val_loss: 0.4480
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 262ms/step - accuracy: 0.8915 - loss: 0.3050 - val_accuracy: 0.9731 - val_loss: 0.0774
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 26s 341ms/step - accuracy: 0.9833 - loss: 0.0586 - val_accuracy: 0.9865 - val_loss: 0.0546
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 67s 757ms/step - accuracy: 0.9943 - loss: 0.0281 - val_accuracy: 0.9865 - val_loss: 0.0527
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 88s 840ms/step - accuracy: 0.9965 - loss: 0.0175 - val_accu